# 🌐 Distributed Caching

**Scale caching across multiple servers**

---

## 📋 Overview

**What you'll learn:**
- Redis clustering
- Cache consistency
- Distributed invalidation
- Partition tolerance
- Scaling strategies

**Time estimate:** ⏱️ 45 minutes | **Difficulty:** 🔴 Advanced

In [ ]:
print("✅ Distributed Caching Setup Complete")

## 🤔 Why Distributed Caching?

### Single Server Problem:

```
Server 1: Has cache entry
Server 2: Cache miss (different memory)
Server 3: Cache miss

❌ Inefficient:
  - No cache sharing
  - 3x API calls for same query
  - Wasted resources
```

### Distributed Solution:

```
All Servers → Shared Redis Cache

✅ Benefits:
  - Shared cache across all servers
  - Higher hit rates
  - Consistent data
  - Scalable
```

## 🔴 Redis Cluster Setup

```python
import redis
from redis.cluster import RedisCluster

# Single Redis instance
cache = redis.Redis(host='localhost', port=6379)

# Redis Cluster (distributed)
cluster = RedisCluster(
    host='localhost',
    port=7000,
    decode_responses=True
)

# Use like normal Redis
cluster.set('key', 'value', ex=3600)
value = cluster.get('key')
```

### Redis Cluster Features:

- **Sharding**: Data split across nodes
- **Replication**: Each shard has replicas
- **Auto-failover**: Automatic recovery
- **No single point of failure**

## 🔄 Cache Invalidation in Distributed Systems

```python
# Pub/Sub for cache invalidation

import redis

class DistributedCache:
    def __init__(self):
        self.redis = redis.Redis()
        self.local_cache = {}  # L1 cache
        
        # Subscribe to invalidation events
        self.pubsub = self.redis.pubsub()
        self.pubsub.subscribe('cache_invalidate')
    
    def get(self, key):
        # Check local cache first
        if key in self.local_cache:
            return self.local_cache[key]
        
        # Check Redis
        value = self.redis.get(key)
        if value:
            self.local_cache[key] = value
        return value
    
    def set(self, key, value, ttl=3600):
        # Update Redis
        self.redis.setex(key, ttl, value)
        
        # Update local cache
        self.local_cache[key] = value
    
    def invalidate(self, key):
        # Remove from Redis
        self.redis.delete(key)
        
        # Notify all servers to invalidate local cache
        self.redis.publish('cache_invalidate', key)
        
        # Remove from local cache
        self.local_cache.pop(key, None)
```

## ✅ Summary

### Key Concepts:

**1. Shared Cache**
```
All servers → Redis Cluster
Benefits: Consistent, scalable, high hit rates
```

**2. Cache Invalidation**
```
Use Pub/Sub for distributed invalidation
Ensure consistency across servers
```

**3. Scaling**
```
Redis Cluster: Automatic sharding
Horizontal scaling: Add more nodes
Replication: High availability
```

### Next: `09_caching/05_cache_warming.ipynb`